# Fase 1 — Carga inicial de los CSV de CICFlowMeter

##### - Cuántos CSV tienes
##### - Cuántas filas aporta cada uno
##### - Cuántas filas y columnas tiene el conjunto total
##### - Qué columnas están disponibles

In [2]:
import os
import pandas as pd

path_cic = "/home/miguel/Escritorio/TFM/TFM_Miguel/ArchivosCIC/BenignTraffic/"

csvs_cic = sorted([f for f in os.listdir(path_cic) if f.endswith(".csv")])

print("CSV encontrados:", len(csvs_cic))
for f in csvs_cic:
    print(f)

lista_dfs = []

for archivo in csvs_cic:
    ruta_completa = os.path.join(path_cic, archivo)
    df_temp = pd.read_csv(ruta_completa)
    df_temp["archivo_origen"] = archivo
    lista_dfs.append(df_temp)
    print(f"{archivo} -> {df_temp.shape}")

df_cic = pd.concat(lista_dfs, ignore_index=True)

print("\nShape total:", df_cic.shape)
print("Número de columnas:", len(df_cic.columns))
print(df_cic.columns.tolist())

df_cic.head()

CSV encontrados: 4
BenignTraffic.pcap_Flow.csv
BenignTraffic1.pcap_Flow.csv
BenignTraffic2.pcap_Flow.csv
BenignTraffic3.pcap_Flow.csv
BenignTraffic.pcap_Flow.csv -> (183630, 85)
BenignTraffic1.pcap_Flow.csv -> (84526, 85)
BenignTraffic2.pcap_Flow.csv -> (91279, 85)
BenignTraffic3.pcap_Flow.csv -> (38895, 85)

Shape total: (398330, 85)
Número de columnas: 85
['Flow ID', 'Src IP', 'Src Port', 'Dst IP', 'Dst Port', 'Protocol', 'Timestamp', 'Flow Duration', 'Total Fwd Packet', 'Total Bwd packets', 'Total Length of Fwd Packet', 'Total Length of Bwd Packet', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max',

,Flow ID,Src IP,Src Port,Dst IP,Dst Port,Protocol,Timestamp,Flow Duration,Total Fwd Packet,Total Bwd packets,...,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label,archivo_origen
0,192.168.137.41-157.249.81.141-51746-80-6,192.168.137.41,51746,157.249.81.141,80,6,07/10/2022 07:15:01 p. m.,291959,5,4,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BenignTraffic,BenignTraffic.pcap_Flow.csv
1,192.168.137.41-157.249.81.141-50096-443-6,192.168.137.41,50096,157.249.81.141,443,6,07/10/2022 07:15:01 p. m.,291320,5,4,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BenignTraffic,BenignTraffic.pcap_Flow.csv
2,192.168.137.41-157.249.81.141-51749-80-6,192.168.137.41,51749,157.249.81.141,80,6,07/10/2022 07:15:03 p. m.,292739,5,4,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BenignTraffic,BenignTraffic.pcap_Flow.csv
3,192.168.137.41-157.249.81.141-50099-443-6,192.168.137.41,50099,157.249.81.141,443,6,07/10/2022 07:15:03 p. m.,292398,5,4,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BenignTraffic,BenignTraffic.pcap_Flow.csv
4,192.168.137.41-157.249.81.141-51752-80-6,192.168.137.41,51752,157.249.81.141,80,6,07/10/2022 07:15:04 p. m.,293252,5,4,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BenignTraffic,BenignTraffic.pcap_Flow.csv


# FASE 2 - Extraer los Flow ID y comprobar si con este unicamente sirven

##### Ver si el Flow ID que trae CICFlowMeter te vale como identificador único o no.
##### Flow ID en CICFlowMeter es la 5-tupla:
##### -Src IP
##### -Dst IP
##### -Src Port
##### -Dst Port
##### -Protocol


In [3]:
print("=== ANÁLISIS DE FLOW ID ===\n")

total_filas = len(df_cic)
flow_id_unicos = df_cic["Flow ID"].nunique()
filas_duplicadas = df_cic.duplicated(subset=["Flow ID"]).sum()

print(f"Total de filas: {total_filas}")
print(f"Flow ID distintos: {flow_id_unicos}")
print(f"Filas duplicadas (mismo Flow ID): {filas_duplicadas} ({(filas_duplicadas/total_filas)*100:.2f}%)")

# Flow ID que aparecen más de 1 vez
conteo_flow = df_cic["Flow ID"].value_counts()
flow_id_repetidos = conteo_flow[conteo_flow > 1]

print(f"\nFlow ID que aparecen más de una vez: {len(flow_id_repetidos)}")

print("\n=== EJEMPLOS DE FLOW ID REPETIDOS ===")
display(flow_id_repetidos.head(10).rename("num_veces"))

# Ver un ejemplo concreto
if len(flow_id_repetidos) > 0:
    ejemplo_flow = flow_id_repetidos.index[0]
    
    print(f"\n=== DETALLE DE UN FLOW REPETIDO ({ejemplo_flow}) ===")
    
    display(
        df_cic[df_cic["Flow ID"] == ejemplo_flow][[
            "Flow ID", "Src IP", "Src Port", "Dst IP", "Dst Port",
            "Protocol", "Timestamp", "Flow Duration", "Label", "archivo_origen"
        ]]
    )

=== ANÁLISIS DE FLOW ID ===

Total de filas: 398330
Flow ID distintos: 239655
Filas duplicadas (mismo Flow ID): 158675 (39.84%)

Flow ID que aparecen más de una vez: 45437

=== EJEMPLOS DE FLOW ID REPETIDOS ===


Flow ID
8.6.0.1-8.0.6.4-0-0-0                            840
192.168.137.192-255.255.255.255-49231-6667-17    840
192.168.137.136-255.255.255.255-49549-6667-17    840
192.168.137.31-255.255.255.255-49287-6667-17     840
192.168.137.224-18.221.14.3-47674-8006-17        838
192.168.137.227-66.23.204.162-45855-10001-17     837
192.168.137.227-147.135.36.150-45855-10001-17    837
192.168.137.154-224.0.0.22-0-0-0                 837
192.168.137.227-192.99.160.133-45855-10001-17    837
192.168.137.148-54.173.75.186-41316-443-6        837
Name: num_veces, dtype: int64


=== DETALLE DE UN FLOW REPETIDO (8.6.0.1-8.0.6.4-0-0-0) ===


,Flow ID,Src IP,Src Port,Dst IP,Dst Port,Protocol,Timestamp,Flow Duration,Label,archivo_origen
217,8.6.0.1-8.0.6.4-0-0-0,8.6.0.1,0,8.0.6.4,0,0,07/10/2022 07:15:00 p. m.,119580202,BenignTraffic,BenignTraffic.pcap_Flow.csv
541,8.6.0.1-8.0.6.4-0-0-0,8.6.0.1,0,8.0.6.4,0,0,07/10/2022 07:17:02 p. m.,119411456,BenignTraffic,BenignTraffic.pcap_Flow.csv
848,8.6.0.1-8.0.6.4-0-0-0,8.6.0.1,0,8.0.6.4,0,0,07/10/2022 07:19:02 p. m.,118398290,BenignTraffic,BenignTraffic.pcap_Flow.csv
1148,8.6.0.1-8.0.6.4-0-0-0,8.6.0.1,0,8.0.6.4,0,0,07/10/2022 07:21:02 p. m.,119105291,BenignTraffic,BenignTraffic.pcap_Flow.csv
1453,8.6.0.1-8.0.6.4-0-0-0,8.6.0.1,0,8.0.6.4,0,0,07/10/2022 07:23:02 p. m.,119813326,BenignTraffic,BenignTraffic.pcap_Flow.csv
...,...,...,...,...,...,...,...,...,...,...
377160,8.6.0.1-8.0.6.4-0-0-0,8.6.0.1,0,8.0.6.4,0,0,08/10/2022 11:05:46 p. m.,119227586,BenignTraffic,BenignTraffic3.pcap_Flow.csv
377331,8.6.0.1-8.0.6.4-0-0-0,8.6.0.1,0,8.0.6.4,0,0,08/10/2022 11:07:47 p. m.,119994950,BenignTraffic,BenignTraffic3.pcap_Flow.csv
377508,8.6.0.1-8.0.6.4-0-0-0,8.6.0.1,0,8.0.6.4,0,0,08/10/2022 11:09:47 p. m.,119971791,BenignTraffic,BenignTraffic3.pcap_Flow.csv
377686,8.6.0.1-8.0.6.4-0-0-0,8.6.0.1,0,8.0.6.4,0,0,08/10/2022 11:11:47 p. m.,119976018,BenignTraffic,BenignTraffic3.pcap_Flow.csv


In [4]:
# ==============================
# ANÁLISIS DE REPETICIONES DE FLOW ID
# ==============================

# 1) Contar cuántas veces aparece cada Flow ID
conteo_flows = df_cic["Flow ID"].value_counts()

# 2) Distribución completa:
#    índice = número de veces que se repite un Flow ID
#    valor  = cuántos Flow ID tienen esa repetición
distribucion = conteo_flows.value_counts().sort_index()

# ==============================
# RESUMEN GENERAL
# ==============================
total_filas = len(df_cic)
total_flows_unicos = conteo_flows.shape[0]
flows_no_repetidos = (conteo_flows == 1).sum()
flows_repetidos = (conteo_flows > 1).sum()
filas_extra = (conteo_flows - 1).clip(lower=0).sum()

print("=== RESUMEN GENERAL ===")
print(f"Total de filas: {total_filas}")
print(f"Total de Flow ID distintos: {total_flows_unicos}")
print(f"Flow ID que aparecen solo 1 vez: {flows_no_repetidos}")
print(f"Flow ID que aparecen más de 1 vez: {flows_repetidos}")
print(f"Filas extra por repeticiones: {filas_extra}")

# ==============================
# REPETICIONES EXACTAS (TODAS)
# ==============================
print("\n=== CUÁNTAS VECES SE REPITE CADA FLOW ID (TODOS) ===")
for veces, cantidad in distribucion.items():
    print(f"{veces} veces -> {cantidad} Flow ID")

# ==============================
# AGRUPACIÓN POR RANGOS
# ==============================
rangos = {
    "1 vez": (conteo_flows == 1).sum(),
    "2 veces": (conteo_flows == 2).sum(),
    "3-5 veces": ((conteo_flows >= 3) & (conteo_flows <= 5)).sum(),
    "6-10 veces": ((conteo_flows >= 6) & (conteo_flows <= 10)).sum(),
    "11-50 veces": ((conteo_flows >= 11) & (conteo_flows <= 50)).sum(),
    "51-100 veces": ((conteo_flows >= 51) & (conteo_flows <= 100)).sum(),
    "Más de 100 veces": (conteo_flows > 100).sum()
}

print("\n=== AGRUPACIÓN POR RANGOS ===")
for rango, cantidad in rangos.items():
    porcentaje = (cantidad / total_flows_unicos) * 100
    print(f"{rango}: {cantidad} Flow ID ({porcentaje:.2f}%)")

# ==============================
# LÍMITES REALES DEL DATASET
# ==============================
print("\n=== LÍMITES DEL DATASET ===")
print(f"Mínimo número de repeticiones: {conteo_flows.min()}")
print(f"Máximo número de repeticiones: {conteo_flows.max()}")
print(f"Número total de valores distintos de repeticiones: {len(distribucion)}")

# ==============================
# COMPROBACIONES
# ==============================
print("\n=== COMPROBACIONES ===")
print(f"Suma de Flow ID: {distribucion.sum()} (debería ser {total_flows_unicos})")

total_filas_calc = (distribucion.index.to_numpy() * distribucion.values).sum()
print(f"Suma de filas: {total_filas_calc} (debería ser {total_filas})")

# ==============================
# TOP 10 FLOW ID MÁS REPETIDOS
# ==============================
print("\n=== TOP 10 FLOW ID MÁS REPETIDOS ===")
print(conteo_flows.head(10).rename("num_veces"))

=== RESUMEN GENERAL ===
Total de filas: 398330
Total de Flow ID distintos: 239655
Flow ID que aparecen solo 1 vez: 194218
Flow ID que aparecen más de 1 vez: 45437
Filas extra por repeticiones: 158675

=== CUÁNTAS VECES SE REPITE CADA FLOW ID (TODOS) ===
1 veces -> 194218 Flow ID
2 veces -> 34952 Flow ID
3 veces -> 7550 Flow ID
4 veces -> 1614 Flow ID
5 veces -> 374 Flow ID
6 veces -> 94 Flow ID
7 veces -> 32 Flow ID
8 veces -> 21 Flow ID
9 veces -> 8 Flow ID
10 veces -> 13 Flow ID
11 veces -> 8 Flow ID
12 veces -> 23 Flow ID
13 veces -> 17 Flow ID
14 veces -> 15 Flow ID
15 veces -> 9 Flow ID
16 veces -> 6 Flow ID
17 veces -> 3 Flow ID
18 veces -> 3 Flow ID
19 veces -> 1 Flow ID
20 veces -> 3 Flow ID
21 veces -> 5 Flow ID
22 veces -> 3 Flow ID
23 veces -> 12 Flow ID
24 veces -> 10 Flow ID
25 veces -> 8 Flow ID
26 veces -> 9 Flow ID
27 veces -> 5 Flow ID
28 veces -> 22 Flow ID
29 veces -> 213 Flow ID
30 veces -> 35 Flow ID
31 veces -> 16 Flow ID
32 veces -> 13 Flow ID
33 veces -> 3 Flow 

##### No es único por sí solo
##### Por tanto, para identificar mejor un flujo hay que incorporar al menos el tiempo (Timestamp), y quizá la duración. (Flow Duration)

# FASE 3 - Limpieza mínima antes de trabajar con identificadores

##### Quedarte solo con columnas que te interesan ahora
##### Revisar cuántos puertos/protocolos 0 hay
##### Filtrar esos casos

In [5]:
# ==============================
# SELECCIÓN DE COLUMNAS BASE
# ==============================

columnas_base = [
    "Flow ID", "Src IP", "Src Port", "Dst IP", "Dst Port",
    "Protocol", "Timestamp", "Flow Duration", "Label", "archivo_origen"
]

df_cic_base = df_cic[columnas_base].copy()

print("Shape original:", df_cic_base.shape)

# ==============================
# ANÁLISIS DE VALORES 0
# ==============================

print("\nValores problemáticos:")
print("Src Port = 0:", (df_cic_base["Src Port"] == 0).sum())
print("Dst Port = 0:", (df_cic_base["Dst Port"] == 0).sum())
print("Protocol = 0:", (df_cic_base["Protocol"] == 0).sum())

# ==============================
# MARCAR FILAS CON CEROS (NO ELIMINAR)
# ==============================

df_cic_base["tiene_ceros"] = (
    (df_cic_base["Src Port"] == 0) |
    (df_cic_base["Dst Port"] == 0) |
    (df_cic_base["Protocol"] == 0)
)

print("\nFilas con valores 0:", df_cic_base["tiene_ceros"].sum())

Shape original: (398330, 10)

Valores problemáticos:
Src Port = 0: 8242
Dst Port = 0: 8242
Protocol = 0: 8242

Filas con valores 0: 8242


# Fase 4 - Arreglar el Timestamp

##### Convertir la fecha/hora de CICFlowMeter a formato datetime, que luego te permitirá ordenar y comparar.

In [6]:
# ==============================
# LIMPIEZA Y NORMALIZACIÓN DE TIMESTAMP
# ==============================

print("Antes:")
print(df_cic_base["Timestamp"].head(3))

# Reemplazar formato AM/PM español
df_cic_base["Timestamp"] = df_cic_base["Timestamp"].str.replace(" p. m.", " PM", regex=False)
df_cic_base["Timestamp"] = df_cic_base["Timestamp"].str.replace(" a. m.", " AM", regex=False)

# Convertir a datetime
df_cic_base["Timestamp"] = pd.to_datetime(
    df_cic_base["Timestamp"],
    format="%d/%m/%Y %I:%M:%S %p",
    errors="coerce"
)

print("\nDespués:")
print(df_cic_base["Timestamp"].head(3))

print("\nValores no convertidos (NaT):", df_cic_base["Timestamp"].isna().sum())

Antes:
0    07/10/2022 07:15:01 p. m.
1    07/10/2022 07:15:01 p. m.
2    07/10/2022 07:15:03 p. m.
Name: Timestamp, dtype: object

Después:
0   2022-10-07 19:15:01
1   2022-10-07 19:15:01
2   2022-10-07 19:15:03
Name: Timestamp, dtype: datetime64[ns]

Valores no convertidos (NaT): 0


# Fase 5 - Construcción del identificador de flujo

##### Crear dos niveles de identificador:

##### uno básico, basado en la 5-tupla
##### otro más fuerte, que incorpore tiempo

In [7]:
df_cic_base["flow_id_full"] = (
    df_cic_base["Flow ID"].astype(str) + "_" +
    df_cic_base["Timestamp"].astype(str)
)


df_cic_base = df_cic_base.sort_values("Timestamp").reset_index(drop=True)

# ==============================
# RESULTADOS
# ==============================
total_filas = len(df_cic_base)
flow_id_unicos = df_cic_base["Flow ID"].nunique()
flow_id_full_unicos = df_cic_base["flow_id_full"].nunique()

print("Filas:", total_filas)
print("Flow ID distintos:", flow_id_unicos)
print("flow_id_full distintos:", flow_id_full_unicos)

# ==============================
# ANÁLISIS DE DUPLICADOS
# ==============================
duplicados_flow_id_full = total_filas - flow_id_full_unicos

print("\n=== ANÁLISIS ===")
print(f"Filas duplicadas incluso con tiempo: {duplicados_flow_id_full}")

# Explicación clara
print("\n=== EXPLICACIÓN ===")
print("Flow ID: identifica la comunicación (IP, puertos, protocolo) → NO es único")
print("flow_id_full: añade el Timestamp → mejora la unicidad")

print("\nInterpretación:")
print(f"- De {total_filas} filas totales, {flow_id_full_unicos} son únicas")
print(f"- Hay {duplicados_flow_id_full} filas que siguen repetidas incluso con Timestamp")

print("\nEsto significa que existen registros con:")
print("- misma IP origen y destino")
print("- mismos puertos")
print("- mismo protocolo")
print("- mismo Timestamp")

Filas: 398330
Flow ID distintos: 239655
flow_id_full distintos: 392195

=== ANÁLISIS ===
Filas duplicadas incluso con tiempo: 6135

=== EXPLICACIÓN ===
Flow ID: identifica la comunicación (IP, puertos, protocolo) → NO es único
flow_id_full: añade el Timestamp → mejora la unicidad

Interpretación:
- De 398330 filas totales, 392195 son únicas
- Hay 6135 filas que siguen repetidas incluso con Timestamp

Esto significa que existen registros con:
- misma IP origen y destino
- mismos puertos
- mismo protocolo
- mismo Timestamp


In [8]:
# ==============================
# ANÁLISIS DE REPETICIONES DE flow_id_full
# ==============================

# 1) Contar cuántas veces aparece cada flow_id_full
conteo_flows_full = df_cic_base["flow_id_full"].value_counts()

# 2) Distribución completa:
#    índice = número de veces que se repite un flow_id_full
#    valor  = cuántos flow_id_full tienen esa repetición
distribucion_full = conteo_flows_full.value_counts().sort_index()

# ==============================
# RESUMEN GENERAL
# ==============================
total_filas = len(df_cic_base)
total_flows_full_distintos = conteo_flows_full.shape[0]
flows_full_no_repetidos = (conteo_flows_full == 1).sum()
flows_full_repetidos = (conteo_flows_full > 1).sum()
filas_extra_full = (conteo_flows_full - 1).clip(lower=0).sum()

print("=== RESUMEN GENERAL ===")
print(f"Total de filas: {total_filas}")
print(f"flow_id_full distintos: {total_flows_full_distintos}")
print(f"flow_id_full que aparecen solo 1 vez: {flows_full_no_repetidos}")
print(f"flow_id_full que aparecen más de 1 vez: {flows_full_repetidos}")
print(f"Filas extra por repeticiones: {filas_extra_full}")

# ==============================
# REPETICIONES EXACTAS (TODAS)
# ==============================
print("\n=== CUÁNTAS VECES SE REPITE CADA flow_id_full (TODOS) ===")
for veces, cantidad in distribucion_full.items():
    print(f"{veces} veces -> {cantidad} flow_id_full")

# ==============================
# AGRUPACIÓN POR RANGOS
# ==============================
rangos_full = {
    "1 vez": (conteo_flows_full == 1).sum(),
    "2 veces": (conteo_flows_full == 2).sum(),
    "3-5 veces": ((conteo_flows_full >= 3) & (conteo_flows_full <= 5)).sum(),
    "6-10 veces": ((conteo_flows_full >= 6) & (conteo_flows_full <= 10)).sum(),
    "11-50 veces": ((conteo_flows_full >= 11) & (conteo_flows_full <= 50)).sum(),
    "51-100 veces": ((conteo_flows_full >= 51) & (conteo_flows_full <= 100)).sum(),
    "Más de 100 veces": (conteo_flows_full > 100).sum()
}

print("\n=== AGRUPACIÓN POR RANGOS ===")
for rango, cantidad in rangos_full.items():
    porcentaje = (cantidad / total_flows_full_distintos) * 100
    print(f"{rango}: {cantidad} flow_id_full ({porcentaje:.2f}%)")

# ==============================
# LÍMITES REALES DEL DATASET
# ==============================
print("\n=== LÍMITES DEL DATASET ===")
print(f"Mínimo número de repeticiones: {conteo_flows_full.min()}")
print(f"Máximo número de repeticiones: {conteo_flows_full.max()}")
print(f"Número total de valores distintos de repeticiones: {len(distribucion_full)}")

# ==============================
# COMPROBACIONES
# ==============================
print("\n=== COMPROBACIONES ===")
print(f"Suma de flow_id_full: {distribucion_full.sum()} (debería ser {total_flows_full_distintos})")

total_filas_calc = (distribucion_full.index.to_numpy() * distribucion_full.values).sum()
print(f"Suma de filas: {total_filas_calc} (debería ser {total_filas})")

# ==============================
# TOP 10 flow_id_full MÁS REPETIDOS
# ==============================
print("\n=== TOP 10 flow_id_full MÁS REPETIDOS ===")
print(conteo_flows_full.head(10).rename("num_veces"))

=== RESUMEN GENERAL ===
Total de filas: 398330
flow_id_full distintos: 392195
flow_id_full que aparecen solo 1 vez: 386410
flow_id_full que aparecen más de 1 vez: 5785
Filas extra por repeticiones: 6135

=== CUÁNTAS VECES SE REPITE CADA flow_id_full (TODOS) ===
1 veces -> 386410 flow_id_full
2 veces -> 5704 flow_id_full
3 veces -> 58 flow_id_full
4 veces -> 2 flow_id_full
5 veces -> 2 flow_id_full
6 veces -> 1 flow_id_full
7 veces -> 3 flow_id_full
8 veces -> 2 flow_id_full
9 veces -> 1 flow_id_full
11 veces -> 1 flow_id_full
12 veces -> 1 flow_id_full
13 veces -> 1 flow_id_full
14 veces -> 1 flow_id_full
18 veces -> 1 flow_id_full
19 veces -> 1 flow_id_full
26 veces -> 1 flow_id_full
27 veces -> 1 flow_id_full
28 veces -> 1 flow_id_full
29 veces -> 1 flow_id_full
35 veces -> 1 flow_id_full
36 veces -> 1 flow_id_full

=== AGRUPACIÓN POR RANGOS ===
1 vez: 386410 flow_id_full (98.52%)
2 veces: 5704 flow_id_full (1.45%)
3-5 veces: 62 flow_id_full (0.02%)
6-10 veces: 7 flow_id_full (0.00%)

In [9]:
# ==============================
# CREACIÓN DE flow_id_full_duration
# ==============================

df_cic_base["flow_id_full_duration"] = (
    df_cic_base["Flow ID"].astype(str) + "_" +
    df_cic_base["Timestamp"].astype(str) + "_" +
    df_cic_base["Flow Duration"].astype(str)
)

# ==============================
# ANÁLISIS DE REPETICIONES DE flow_id_full_duration
# ==============================

# 1) Contar cuántas veces aparece cada flow_id_full_duration
conteo_flows_full_duration = df_cic_base["flow_id_full_duration"].value_counts()

# 2) Distribución completa:
#    índice = número de veces que se repite un flow_id_full_duration
#    valor  = cuántos flow_id_full_duration tienen esa repetición
distribucion_full_duration = conteo_flows_full_duration.value_counts().sort_index()

# ==============================
# RESUMEN GENERAL
# ==============================
total_filas = len(df_cic_base)
total_flows_full_duration_distintos = conteo_flows_full_duration.shape[0]
flows_full_duration_no_repetidos = (conteo_flows_full_duration == 1).sum()
flows_full_duration_repetidos = (conteo_flows_full_duration > 1).sum()
filas_extra_full_duration = (conteo_flows_full_duration - 1).clip(lower=0).sum()

print("=== RESUMEN GENERAL ===")
print(f"Total de filas: {total_filas}")
print(f"flow_id_full_duration distintos: {total_flows_full_duration_distintos}")
print(f"flow_id_full_duration que aparecen solo 1 vez: {flows_full_duration_no_repetidos}")
print(f"flow_id_full_duration que aparecen más de 1 vez: {flows_full_duration_repetidos}")
print(f"Filas extra por repeticiones: {filas_extra_full_duration}")

# ==============================
# REPETICIONES EXACTAS (TODAS)
# ==============================
print("\n=== CUÁNTAS VECES SE REPITE CADA flow_id_full_duration (TODOS) ===")
for veces, cantidad in distribucion_full_duration.items():
    print(f"{veces} veces -> {cantidad} flow_id_full_duration")

# ==============================
# AGRUPACIÓN POR RANGOS
# ==============================
rangos_full_duration = {
    "1 vez": (conteo_flows_full_duration == 1).sum(),
    "2 veces": (conteo_flows_full_duration == 2).sum(),
    "3-5 veces": ((conteo_flows_full_duration >= 3) & (conteo_flows_full_duration <= 5)).sum(),
    "6-10 veces": ((conteo_flows_full_duration >= 6) & (conteo_flows_full_duration <= 10)).sum(),
    "11-50 veces": ((conteo_flows_full_duration >= 11) & (conteo_flows_full_duration <= 50)).sum(),
    "51-100 veces": ((conteo_flows_full_duration >= 51) & (conteo_flows_full_duration <= 100)).sum(),
    "Más de 100 veces": (conteo_flows_full_duration > 100).sum()
}

print("\n=== AGRUPACIÓN POR RANGOS ===")
for rango, cantidad in rangos_full_duration.items():
    porcentaje = (cantidad / total_flows_full_duration_distintos) * 100
    print(f"{rango}: {cantidad} flow_id_full_duration ({porcentaje:.2f}%)")

# ==============================
# LÍMITES REALES DEL DATASET
# ==============================
print("\n=== LÍMITES DEL DATASET ===")
print(f"Mínimo número de repeticiones: {conteo_flows_full_duration.min()}")
print(f"Máximo número de repeticiones: {conteo_flows_full_duration.max()}")
print(f"Número total de valores distintos de repeticiones: {len(distribucion_full_duration)}")

# ==============================
# COMPROBACIONES
# ==============================
print("\n=== COMPROBACIONES ===")
print(f"Suma de flow_id_full_duration: {distribucion_full_duration.sum()} (debería ser {total_flows_full_duration_distintos})")

total_filas_calc = (distribucion_full_duration.index.to_numpy() * distribucion_full_duration.values).sum()
print(f"Suma de filas: {total_filas_calc} (debería ser {total_filas})")

# ==============================
# TOP 10 flow_id_full_duration MÁS REPETIDOS
# ==============================
print("\n=== TOP 10 flow_id_full_duration MÁS REPETIDOS ===")
print(conteo_flows_full_duration.head(10).rename("num_veces"))

=== RESUMEN GENERAL ===
Total de filas: 398330
flow_id_full_duration distintos: 398252
flow_id_full_duration que aparecen solo 1 vez: 398179
flow_id_full_duration que aparecen más de 1 vez: 73
Filas extra por repeticiones: 78

=== CUÁNTAS VECES SE REPITE CADA flow_id_full_duration (TODOS) ===
1 veces -> 398179 flow_id_full_duration
2 veces -> 71 flow_id_full_duration
4 veces -> 1 flow_id_full_duration
5 veces -> 1 flow_id_full_duration

=== AGRUPACIÓN POR RANGOS ===
1 vez: 398179 flow_id_full_duration (99.98%)
2 veces: 71 flow_id_full_duration (0.02%)
3-5 veces: 2 flow_id_full_duration (0.00%)
6-10 veces: 0 flow_id_full_duration (0.00%)
11-50 veces: 0 flow_id_full_duration (0.00%)
51-100 veces: 0 flow_id_full_duration (0.00%)
Más de 100 veces: 0 flow_id_full_duration (0.00%)

=== LÍMITES DEL DATASET ===
Mínimo número de repeticiones: 1
Máximo número de repeticiones: 5
Número total de valores distintos de repeticiones: 4

=== COMPROBACIONES ===
Suma de flow_id_full_duration: 398252 (deb

In [10]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 300)

display(
    df_cic_base[[
        "Flow ID", "Timestamp", "Flow Duration", "flow_id_full_duration"
    ]].head(10)
)

,Flow ID,Timestamp,Flow Duration,flow_id_full_duration
0,192.168.137.175-99.81.244.93-56891-443-6,2022-10-07 19:15:00,42165987,192.168.137.175-99.81.244.93-56891-443-6_2022-10-07 19:15:00_42165987
1,192.168.137.7-255.255.255.255-49154-6667-17,2022-10-07 19:15:00,115003544,192.168.137.7-255.255.255.255-49154-6667-17_2022-10-07 19:15:00_115003544
2,192.168.137.227-192.99.160.133-45855-10001-17,2022-10-07 19:15:00,110642000,192.168.137.227-192.99.160.133-45855-10001-17_2022-10-07 19:15:00_110642000
3,192.168.137.227-147.135.36.150-45855-10001-17,2022-10-07 19:15:00,110641953,192.168.137.227-147.135.36.150-45855-10001-17_2022-10-07 19:15:00_110641953
4,192.168.137.227-66.23.204.162-45855-10001-17,2022-10-07 19:15:00,110642028,192.168.137.227-66.23.204.162-45855-10001-17_2022-10-07 19:15:00_110642028
5,192.168.137.172-192.168.137.1-0-0-0,2022-10-07 19:15:00,4005653,192.168.137.172-192.168.137.1-0-0-0_2022-10-07 19:15:00_4005653
6,192.168.137.46-35.185.101.66-48335-443-6,2022-10-07 19:15:00,119932388,192.168.137.46-35.185.101.66-48335-443-6_2022-10-07 19:15:00_119932388
7,8.6.0.1-8.0.6.4-0-0-0,2022-10-07 19:15:00,119580202,8.6.0.1-8.0.6.4-0-0-0_2022-10-07 19:15:00_119580202
8,192.168.137.167-255.255.255.255-49229-6667-17,2022-10-07 19:15:00,116717343,192.168.137.167-255.255.255.255-49229-6667-17_2022-10-07 19:15:00_116717343
9,192.168.137.82-255.255.255.255-49236-6667-17,2022-10-07 19:15:00,117098772,192.168.137.82-255.255.255.255-49236-6667-17_2022-10-07 19:15:00_117098772


In [11]:
print("PRIMERA FILA DE df_cic")
print(df_cic.loc[0, ["Flow ID", "Timestamp", "Flow Duration"]])

print("\nPRIMERA FILA DE df_cic_base")
print(df_cic_base.loc[0, ["Flow ID", "Timestamp", "Flow Duration", "flow_id_full_duration"]])

PRIMERA FILA DE df_cic
Flow ID          192.168.137.41-157.249.81.141-51746-80-6
Timestamp                       07/10/2022 07:15:01 p. m.
Flow Duration                                      291959
Name: 0, dtype: object

PRIMERA FILA DE df_cic_base
Flow ID                                               192.168.137.175-99.81.244.93-56891-443-6
Timestamp                                                                  2022-10-07 19:15:00
Flow Duration                                                                         42165987
flow_id_full_duration    192.168.137.175-99.81.244.93-56891-443-6_2022-10-07 19:15:00_42165987
Name: 0, dtype: object


In [12]:
flow = "192.168.137.41-157.249.81.141-51746-80-6"

print("En df_cic:")
display(df_cic[df_cic["Flow ID"] == flow][["Flow ID", "Timestamp", "Flow Duration"]].head())

print("En df_cic_base:")
display(df_cic_base[df_cic_base["Flow ID"] == flow][["Flow ID", "Timestamp", "Flow Duration", "flow_id_full_duration"]].head())

En df_cic:


,Flow ID,Timestamp,Flow Duration
0,192.168.137.41-157.249.81.141-51746-80-6,07/10/2022 07:15:01 p. m.,291959


En df_cic_base:


,Flow ID,Timestamp,Flow Duration,flow_id_full_duration
24,192.168.137.41-157.249.81.141-51746-80-6,2022-10-07 19:15:01,291959,192.168.137.41-157.249.81.141-51746-80-6_2022-10-07 19:15:01_291959


In [13]:
# 1) Crear identificador base en df_cic ORIGINAL
df_cic["id_base"] = (
    df_cic["Flow ID"].astype(str) + "_" +
    df_cic["Timestamp"].astype(str) + "_" +
    df_cic["Flow Duration"].astype(str)
)

# 2) Obtener los que se repiten
conteo = df_cic["id_base"].value_counts()
ids_repetidos = conteo[conteo > 1].index

# 3) Filtrar esos casos
df_repetidos = df_cic[df_cic["id_base"].isin(ids_repetidos)].copy()

print("Filas repetidas:", len(df_repetidos))

Filas repetidas: 151


In [14]:
# ==============================
# COMPARACIÓN DE IDENTIFICADORES
# ==============================

df_cic_base["flow_id_full"] = (
    df_cic_base["Flow ID"].astype(str) + "_" +
    df_cic_base["Timestamp"].astype(str)
)

df_cic_base["flow_id_full_duration"] = (
    df_cic_base["Flow ID"].astype(str) + "_" +
    df_cic_base["Timestamp"].astype(str) + "_" +
    df_cic_base["Flow Duration"].astype(str)
)

total_filas = len(df_cic_base)

# ==============================
# CÁLCULOS
# ==============================

resultados = {
    "Flow ID": df_cic_base["Flow ID"].nunique(),
    "Flow ID + Timestamp": df_cic_base["flow_id_full"].nunique(),
    "Flow ID + Timestamp + Duration": df_cic_base["flow_id_full_duration"].nunique()
}

# ==============================
# RESULTADOS
# ==============================

print("=== COMPARACIÓN DE UNICIDAD ===\n")
print(f"Total de filas: {total_filas}\n")

for nombre, distintos in resultados.items():
    duplicados = total_filas - distintos
    porcentaje = (duplicados / total_filas) * 100
    
    print(f"{nombre}:")
    print(f"  - IDs distintos: {distintos}")
    print(f"  - Filas no únicas: {duplicados} ({porcentaje:.4f}%)\n")

# ==============================
# CONCLUSIÓN
# ==============================

print("=== CONCLUSIÓN ===")
print("Añadir información progresivamente mejora la unicidad:")
print("- Flow ID → NO suficiente")
print("- + Timestamp → mejora notable")
print("- + Duration → casi unicidad total")

# ==============================
# CASOS RESIDUALES
# ==============================

duplicados_finales = df_cic_base[
    df_cic_base.duplicated("flow_id_full_duration", keep=False)
]

print(f"\nCasos no únicos finales: {len(duplicados_finales)}")

if len(duplicados_finales) > 0:
    display(
        duplicados_finales[[
            "Flow ID", "Timestamp", "Flow Duration", "Label"
        ]].head(10)
    )

=== COMPARACIÓN DE UNICIDAD ===

Total de filas: 398330

Flow ID:
  - IDs distintos: 239655
  - Filas no únicas: 158675 (39.8351%)

Flow ID + Timestamp:
  - IDs distintos: 392195
  - Filas no únicas: 6135 (1.5402%)

Flow ID + Timestamp + Duration:
  - IDs distintos: 398252
  - Filas no únicas: 78 (0.0196%)

=== CONCLUSIÓN ===
Añadir información progresivamente mejora la unicidad:
- Flow ID → NO suficiente
- + Timestamp → mejora notable
- + Duration → casi unicidad total

Casos no únicos finales: 151


,Flow ID,Timestamp,Flow Duration,Label
87278,192.168.137.207-74.125.0.107-56606-80-6,2022-10-07 23:09:49,2,BenignTraffic
87283,192.168.137.207-74.125.0.107-56606-80-6,2022-10-07 23:09:49,2,BenignTraffic
87323,192.168.137.207-74.125.0.107-56606-80-6,2022-10-07 23:09:49,254,BenignTraffic
87342,192.168.137.207-74.125.0.107-56606-80-6,2022-10-07 23:09:49,2,BenignTraffic
87347,192.168.137.207-74.125.0.107-56606-80-6,2022-10-07 23:09:49,2,BenignTraffic
87362,192.168.137.207-74.125.0.107-56606-80-6,2022-10-07 23:09:49,2,BenignTraffic
87363,192.168.137.207-74.125.0.107-56606-80-6,2022-10-07 23:09:49,254,BenignTraffic
96591,192.168.137.253-173.198.192.101-36780-4431-6,2022-10-07 23:34:55,1,BenignTraffic
96596,192.168.137.253-173.198.192.101-36780-4431-6,2022-10-07 23:34:55,1,BenignTraffic
101727,192.168.137.253-173.198.192.101-37022-4431-6,2022-10-07 23:47:56,2,BenignTraffic


# NEMEA

In [28]:
import os
import pandas as pd
import numpy as np

# ==============================
# 1. CARGAR CIC (4 CSV)
# ==============================
path_cic = "/home/miguel/Escritorio/TFM/TFM_Miguel/ArchivosCIC/BenignTraffic/"

csvs_cic = sorted([f for f in os.listdir(path_cic) if f.endswith(".csv")])

print("CSV de CIC encontrados:", len(csvs_cic))
for f in csvs_cic:
    print(" -", f)

lista_dfs = []

for archivo in csvs_cic:
    ruta_completa = os.path.join(path_cic, archivo)
    df_temp = pd.read_csv(ruta_completa)
    df_temp["archivo_origen"] = archivo
    lista_dfs.append(df_temp)
    print(f"{archivo} -> {df_temp.shape}")

df_cic = pd.concat(lista_dfs, ignore_index=True)

print("\nShape total CIC:", df_cic.shape)
print("Filas CIC:", len(df_cic))


# ==============================
# 2. CARGAR NEMEA
# ==============================
ruta_nemea = "/home/miguel/Escritorio/TFM/TFM_Miguel/ArchivosNEMEA/Benign/BenignTraffic__Total_Expanded.csv"

df_nemea = pd.read_csv(ruta_nemea)

print("\nShape total NEMEA:", df_nemea.shape)
print("Filas NEMEA:", len(df_nemea))

print("\nColumnas CIC:")
print(df_cic.columns.tolist())

print("\nColumnas NEMEA:")
print(df_nemea.columns.tolist())

CSV de CIC encontrados: 4
 - BenignTraffic.pcap_Flow.csv
 - BenignTraffic1.pcap_Flow.csv
 - BenignTraffic2.pcap_Flow.csv
 - BenignTraffic3.pcap_Flow.csv
BenignTraffic.pcap_Flow.csv -> (183630, 85)
BenignTraffic1.pcap_Flow.csv -> (84526, 85)
BenignTraffic2.pcap_Flow.csv -> (91279, 85)
BenignTraffic3.pcap_Flow.csv -> (38895, 85)

Shape total CIC: (398330, 85)
Filas CIC: 398330


/tmp/ipykernel_18305/1342683035.py:36: DtypeWarning: Columns (33,35,36) have mixed types. Specify dtype option on import or set low_memory=False.
  df_nemea = pd.read_csv(ruta_nemea)



Shape total NEMEA: (358565, 200)
Filas NEMEA: 358565

Columnas CIC:
['Flow ID', 'Src IP', 'Src Port', 'Dst IP', 'Dst Port', 'Protocol', 'Timestamp', 'Flow Duration', 'Total Fwd Packet', 'Total Bwd packets', 'Total Length of Fwd Packet', 'Total Length of Bwd Packet', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Packet Length Min', 'Packet Length Max', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'S

In [37]:
# ==============================
# VER FORMATOS DE TIEMPO (PRIMERAS FILAS)
# ==============================

print("\n=== FORMATO CIC (Timestamp) ===")
print(df_cic[["Timestamp"]].head(5))

print("\n=== FORMATO NEMEA (TIME_FIRST y TIME_LAST) ===")
print(df_nemea[["time TIME_FIRST", "time TIME_LAST"]].head(5))


=== FORMATO CIC (Timestamp) ===
                   Timestamp
0  07/10/2022 07:15:01 p. m.
1  07/10/2022 07:15:01 p. m.
2  07/10/2022 07:15:03 p. m.
3  07/10/2022 07:15:03 p. m.
4  07/10/2022 07:15:04 p. m.

=== FORMATO NEMEA (TIME_FIRST y TIME_LAST) ===
              time TIME_FIRST              time TIME_LAST
0  2022-10-07T17:15:06.925906  2022-10-07T17:15:06.975433
1  2022-10-07T17:15:06.992699  2022-10-07T17:15:07.014988
2  2022-10-07T17:15:07.161238  2022-10-07T17:15:07.220728
3  2022-10-07T17:15:08.669483  2022-10-07T17:15:10.071821
4  2022-10-07T17:15:11.050546  2022-10-07T17:15:11.072647


In [ ]:
# ==============================
# CONVERSIÓN A DATETIME (SIN AJUSTES)
# ==============================

# ---- CIC ----
df_cic["Timestamp_dt"] = pd.to_datetime(
    df_cic["Timestamp"]
        .astype(str)
        .str.replace("p. m.", "PM", regex=False)
        .str.replace("a. m.", "AM", regex=False),
    format="%d/%m/%Y %I:%M:%S %p",
    errors="coerce"
)

print("\nCIC - NaT (errores de conversión):", df_cic["Timestamp_dt"].isna().sum())


# ---- NEMEA ----
df_nemea["TIME_FIRST_dt"] = pd.to_datetime(df_nemea["time TIME_FIRST"], errors="coerce")
df_nemea["TIME_LAST_dt"]  = pd.to_datetime(df_nemea["time TIME_LAST"], errors="coerce")

print("NEMEA TIME_FIRST NaT:", df_nemea["TIME_FIRST_dt"].isna().sum())
print("NEMEA TIME_LAST NaT :", df_nemea["TIME_LAST_dt"].isna().sum())

print("\n=== COMPROBACIÓN VISUAL ===")

print(df_cic[["Timestamp", "Timestamp_dt"]].head(5))

print(df_nemea[[
    "time TIME_FIRST", "TIME_FIRST_dt",
    "time TIME_LAST",  "TIME_LAST_dt"]].head(5))


CIC - NaT (errores de conversión): 0
NEMEA TIME_FIRST NaT: 0
NEMEA TIME_LAST NaT : 0

=== COMPROBACIÓN VISUAL ===
                   Timestamp        Timestamp_dt
0  07/10/2022 07:15:01 p. m. 2022-10-07 19:15:01
1  07/10/2022 07:15:01 p. m. 2022-10-07 19:15:01
2  07/10/2022 07:15:03 p. m. 2022-10-07 19:15:03
3  07/10/2022 07:15:03 p. m. 2022-10-07 19:15:03
4  07/10/2022 07:15:04 p. m. 2022-10-07 19:15:04
              time TIME_FIRST              TIME_FIRST_dt              time TIME_LAST               TIME_LAST_dt
0  2022-10-07T17:15:06.925906 2022-10-07 17:15:06.925906  2022-10-07T17:15:06.975433 2022-10-07 17:15:06.975433
1  2022-10-07T17:15:06.992699 2022-10-07 17:15:06.992699  2022-10-07T17:15:07.014988 2022-10-07 17:15:07.014988
2  2022-10-07T17:15:07.161238 2022-10-07 17:15:07.161238  2022-10-07T17:15:07.220728 2022-10-07 17:15:07.220728
3  2022-10-07T17:15:08.669483 2022-10-07 17:15:08.669483  2022-10-07T17:15:10.071821 2022-10-07 17:15:10.071821
4  2022-10-07T17:15:11.050546 2

In [40]:
print("\n=== RANGO TEMPORAL CIC ===")
print("Min CIC:", df_cic["Timestamp_dt"].min())
print("Max CIC:", df_cic["Timestamp_dt"].max())

print("\n=== RANGO TEMPORAL NEMEA ===")
print("Min NEMEA TIME_FIRST:", df_nemea["TIME_FIRST_dt"].min())
print("Max NEMEA TIME_LAST :", df_nemea["TIME_LAST_dt"].max())


=== RANGO TEMPORAL CIC ===
Min CIC: 2022-10-07 19:15:00
Max CIC: 2022-10-08 23:15:00

=== RANGO TEMPORAL NEMEA ===
Min NEMEA TIME_FIRST: 2022-10-07 17:15:00.350417
Max NEMEA TIME_LAST : 2022-10-08 21:15:01.072457


In [42]:
# ==============================
# 1. AJUSTAR NEMEA A UTC+2
# ==============================
df_nemea["TIME_FIRST_utc2"] = df_nemea["TIME_FIRST_dt"] + pd.Timedelta(hours=2)
df_nemea["TIME_LAST_utc2"]  = df_nemea["TIME_LAST_dt"] + pd.Timedelta(hours=2)

print("\n=== COMPROBACIÓN AJUSTE NEMEA +2h ===")
print(df_nemea[[
    "time TIME_FIRST", "TIME_FIRST_dt", "TIME_FIRST_utc2",
    "time TIME_LAST", "TIME_LAST_dt", "TIME_LAST_utc2"
]].head(5))


=== COMPROBACIÓN AJUSTE NEMEA +2h ===
              time TIME_FIRST              TIME_FIRST_dt            TIME_FIRST_utc2              time TIME_LAST               TIME_LAST_dt             TIME_LAST_utc2
0  2022-10-07T17:15:06.925906 2022-10-07 17:15:06.925906 2022-10-07 19:15:06.925906  2022-10-07T17:15:06.975433 2022-10-07 17:15:06.975433 2022-10-07 19:15:06.975433
1  2022-10-07T17:15:06.992699 2022-10-07 17:15:06.992699 2022-10-07 19:15:06.992699  2022-10-07T17:15:07.014988 2022-10-07 17:15:07.014988 2022-10-07 19:15:07.014988
2  2022-10-07T17:15:07.161238 2022-10-07 17:15:07.161238 2022-10-07 19:15:07.161238  2022-10-07T17:15:07.220728 2022-10-07 17:15:07.220728 2022-10-07 19:15:07.220728
3  2022-10-07T17:15:08.669483 2022-10-07 17:15:08.669483 2022-10-07 19:15:08.669483  2022-10-07T17:15:10.071821 2022-10-07 17:15:10.071821 2022-10-07 19:15:10.071821
4  2022-10-07T17:15:11.050546 2022-10-07 17:15:11.050546 2022-10-07 19:15:11.050546  2022-10-07T17:15:11.072647 2022-10-07 17:15:11

In [43]:
# ==============================
# 1. SEGUNDOS AUXILIARES
# ==============================
df_cic["Timestamp_segundo"] = df_cic["Timestamp_dt"].dt.floor("s")

df_nemea["TIME_FIRST_segundo_utc2"] = df_nemea["TIME_FIRST_utc2"].dt.floor("s")
df_nemea["TIME_LAST_segundo_utc2"]  = df_nemea["TIME_LAST_utc2"].dt.floor("s")

print("\n=== COMPROBACIÓN SEGUNDOS ===")
print(df_cic[["Timestamp", "Timestamp_dt", "Timestamp_segundo"]].head(5))

print(df_nemea[[
    "TIME_FIRST_utc2", "TIME_FIRST_segundo_utc2",
    "TIME_LAST_utc2", "TIME_LAST_segundo_utc2"
]].head(5))


=== COMPROBACIÓN SEGUNDOS ===
                   Timestamp        Timestamp_dt   Timestamp_segundo
0  07/10/2022 07:15:01 p. m. 2022-10-07 19:15:01 2022-10-07 19:15:01
1  07/10/2022 07:15:01 p. m. 2022-10-07 19:15:01 2022-10-07 19:15:01
2  07/10/2022 07:15:03 p. m. 2022-10-07 19:15:03 2022-10-07 19:15:03
3  07/10/2022 07:15:03 p. m. 2022-10-07 19:15:03 2022-10-07 19:15:03
4  07/10/2022 07:15:04 p. m. 2022-10-07 19:15:04 2022-10-07 19:15:04
             TIME_FIRST_utc2 TIME_FIRST_segundo_utc2             TIME_LAST_utc2 TIME_LAST_segundo_utc2
0 2022-10-07 19:15:06.925906     2022-10-07 19:15:06 2022-10-07 19:15:06.975433    2022-10-07 19:15:06
1 2022-10-07 19:15:06.992699     2022-10-07 19:15:06 2022-10-07 19:15:07.014988    2022-10-07 19:15:07
2 2022-10-07 19:15:07.161238     2022-10-07 19:15:07 2022-10-07 19:15:07.220728    2022-10-07 19:15:07
3 2022-10-07 19:15:08.669483     2022-10-07 19:15:08 2022-10-07 19:15:10.071821    2022-10-07 19:15:10
4 2022-10-07 19:15:11.050546     2022-10

In [44]:
# ==============================
# 2. CONJUNTO DE SEGUNDOS DE CIC
# ==============================
segundos_cic = set(df_cic["Timestamp_segundo"].dropna().unique())

print("Número de segundos distintos en CIC:", len(segundos_cic))

Número de segundos distintos en CIC: 87296


In [45]:
# ==============================
# 3. MARCAR COINCIDENCIAS EN NEMEA
# ==============================
df_nemea["match_time_first_con_cic"] = df_nemea["TIME_FIRST_segundo_utc2"].isin(segundos_cic)
df_nemea["match_time_last_con_cic"]  = df_nemea["TIME_LAST_segundo_utc2"].isin(segundos_cic)

df_nemea["match_ambos_con_cic"] = (
    df_nemea["match_time_first_con_cic"] &
    df_nemea["match_time_last_con_cic"]
)

df_nemea["match_al_menos_uno_con_cic"] = (
    df_nemea["match_time_first_con_cic"] |
    df_nemea["match_time_last_con_cic"]
)

In [47]:
# ==============================
# 4. RESUMEN GENERAL
# ==============================
n_first = df_nemea["match_time_first_con_cic"].sum()
n_last = df_nemea["match_time_last_con_cic"].sum()
n_ambos = df_nemea["match_ambos_con_cic"].sum()
n_uno = df_nemea["match_al_menos_uno_con_cic"].sum()
total_nemea = len(df_nemea)

print("\n=== RESUMEN NEMEA vs CIC ===")
print("Total filas NEMEA:", total_nemea)
print("Filas NEMEA con TIME_FIRST en segundo presente en CIC:", n_first)
print("Filas NEMEA con TIME_LAST  en segundo presente en CIC:", n_last)
print("Filas NEMEA con BOTH (TIME_FIRST y TIME_LAST) en segundos presentes en CIC:", n_ambos)
print("Filas NEMEA con AL MENOS UNO de los dos en segundos presentes en CIC:", n_uno)

print("\nPorcentajes:")
print(f"TIME_FIRST: {100*n_first/total_nemea:.2f}%")
print(f"TIME_LAST : {100*n_last/total_nemea:.2f}%")
print(f"AMBOS     : {100*n_ambos/total_nemea:.2f}%")
print(f"AL MENOS UNO: {100*n_uno/total_nemea:.2f}%")


=== RESUMEN NEMEA vs CIC ===
Total filas NEMEA: 358565
Filas NEMEA con TIME_FIRST en segundo presente en CIC: 351107
Filas NEMEA con TIME_LAST  en segundo presente en CIC: 342004
Filas NEMEA con BOTH (TIME_FIRST y TIME_LAST) en segundos presentes en CIC: 338404
Filas NEMEA con AL MENOS UNO de los dos en segundos presentes en CIC: 354707

Porcentajes:
TIME_FIRST: 97.92%
TIME_LAST : 95.38%
AMBOS     : 94.38%
AL MENOS UNO: 98.92%


In [49]:
# ==============================
# 5. CONTEO POR SEGUNDO
# ==============================

conteo_first = (
    df_nemea[df_nemea["match_time_first_con_cic"]]
    .groupby("TIME_FIRST_segundo_utc2")
    .size()
    .reset_index(name="num_filas_por_TIME_FIRST")
    .rename(columns={"TIME_FIRST_segundo_utc2": "segundo"})
)

conteo_last = (
    df_nemea[df_nemea["match_time_last_con_cic"]]
    .groupby("TIME_LAST_segundo_utc2")
    .size()
    .reset_index(name="num_filas_por_TIME_LAST")
    .rename(columns={"TIME_LAST_segundo_utc2": "segundo"})
)

conteo_ambos = (
    df_nemea[df_nemea["match_ambos_con_cic"]]
    .groupby("TIME_FIRST_segundo_utc2")
    .size()
    .reset_index(name="num_filas_con_ambos")
    .rename(columns={"TIME_FIRST_segundo_utc2": "segundo"})
)

resumen_por_segundo = (
    conteo_first
    .merge(conteo_last, on="segundo", how="outer")
    .merge(conteo_ambos, on="segundo", how="outer")
    .fillna(0)
    .sort_values("segundo")
)

for col in ["num_filas_por_TIME_FIRST", "num_filas_por_TIME_LAST", "num_filas_con_ambos"]:
    resumen_por_segundo[col] = resumen_por_segundo[col].astype(int)

print("\n=== RESUMEN POR SEGUNDO ===")
print(resumen_por_segundo.head(20))


=== RESUMEN POR SEGUNDO ===
               segundo  num_filas_por_TIME_FIRST  num_filas_por_TIME_LAST  num_filas_con_ambos
0  2022-10-07 19:15:00                        19                        1                   19
1  2022-10-07 19:15:01                        17                        5                   17
2  2022-10-07 19:15:02                        20                       12                   20
3  2022-10-07 19:15:03                        23                       12                   21
4  2022-10-07 19:15:04                        12                        4                   12
5  2022-10-07 19:15:05                         8                        5                    8
6  2022-10-07 19:15:06                        17                        8                   17
7  2022-10-07 19:15:07                         8                        9                    8
8  2022-10-07 19:15:08                         7                        6                    7
9  2022-10-07 19:15:0

# TSHARK

In [2]:
import os
import pandas as pd

path_cic = "/home/miguel/Escritorio/TFM/TFM_Miguel/ArchivosCIC/BenignTraffic/"

csvs_cic = sorted([f for f in os.listdir(path_cic) if f.endswith(".csv")])

print("CSV encontrados:", len(csvs_cic))
for f in csvs_cic:
    print(f)

lista_dfs = []

for archivo in csvs_cic:
    ruta_completa = os.path.join(path_cic, archivo)
    df_temp = pd.read_csv(ruta_completa)
    df_temp["archivo_origen"] = archivo
    lista_dfs.append(df_temp)
    print(f"{archivo} -> {df_temp.shape}")

df_cic = pd.concat(lista_dfs, ignore_index=True)

print("\nShape total:", df_cic.shape)
print("Número de columnas:", len(df_cic.columns))
print(df_cic.columns.tolist())

df_cic.head()

# Tshark
ruta_tshark = "/home/miguel/Escritorio/TFM/TFM_Miguel/ArchivosTshark/Csv/BenignTraffic.csv"
df_tshark = pd.read_csv(ruta_tshark)

CSV encontrados: 4
BenignTraffic.pcap_Flow.csv
BenignTraffic1.pcap_Flow.csv
BenignTraffic2.pcap_Flow.csv
BenignTraffic3.pcap_Flow.csv
BenignTraffic.pcap_Flow.csv -> (183630, 85)
BenignTraffic1.pcap_Flow.csv -> (84526, 85)
BenignTraffic2.pcap_Flow.csv -> (91279, 85)
BenignTraffic3.pcap_Flow.csv -> (38895, 85)

Shape total: (398330, 85)
Número de columnas: 85
['Flow ID', 'Src IP', 'Src Port', 'Dst IP', 'Dst Port', 'Protocol', 'Timestamp', 'Flow Duration', 'Total Fwd Packet', 'Total Bwd packets', 'Total Length of Fwd Packet', 'Total Length of Bwd Packet', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max',

In [6]:
print(df_tshark.columns.tolist())





['srcport.std', 'dstport.std', 'frame.len.min', 'frame.len.max', 'frame.len.std', 'frame.len.mean', 'frame.len.rate', 'ip.flags.rb', 'ip.flags.df', 'ip.flags.mf', 'tcp.flags.res', 'tcp.flags.ns', 'tcp.flags.cwr', 'tcp.flags.ecn', 'tcp.flags.urg', 'tcp.flags.ack', 'tcp.flags.push', 'tcp.flags.reset', 'tcp.flags.syn', 'tcp.flags.fin', 'int.std', 'int.min', 'int.max', 'int.mean', 'count', 'ip.ttl.std', 'ip.ttl.min', 'ip.ttl.max', 'ip.ttl.mean', 'ip.checksum.status.std', 'ip.checksum.status.min', 'ip.checksum.status.max', 'ip.checksum.status.mean', 'l4.checksum.status.std', 'l4.checksum.status.min', 'l4.checksum.status.max', 'l4.checksum.status.mean', 'tcp.seq_raw.std', 'tcp.seq_raw.min', 'tcp.seq_raw.max', 'tcp.seq_raw.mean', 'tcp.ack_raw.std', 'tcp.ack_raw.min', 'tcp.ack_raw.max', 'tcp.ack_raw.mean', 'tcp.window_size_value.std', 'tcp.window_size_value.min', 'tcp.window_size_value.max', 'tcp.window_size_value.mean', 'duration', 'prate', 'payload.std', 'payload.min', 'payload.max', 'payloa

# TSTAT

In [10]:
import os
import pandas as pd

path_cic = "/home/miguel/Escritorio/TFM/TFM_Miguel/ArchivosCIC/BenignTraffic/"

csvs_cic = sorted([f for f in os.listdir(path_cic) if f.endswith(".csv")])

print("CSV encontrados:", len(csvs_cic))
for f in csvs_cic:
    print(f)

lista_dfs = []

for archivo in csvs_cic:
    ruta_completa = os.path.join(path_cic, archivo)
    df_temp = pd.read_csv(ruta_completa)
    df_temp["archivo_origen"] = archivo
    lista_dfs.append(df_temp)
    print(f"{archivo} -> {df_temp.shape}")

df_cic = pd.concat(lista_dfs, ignore_index=True)

print("\nShape total:", df_cic.shape)
print("Número de columnas:", len(df_cic.columns))
print(df_cic.columns.tolist())

df_cic.head()

# Tstat
ruta_tstat = "/home/miguel/Escritorio/TFM/TFM_Miguel/ArchivosTstat/CSV/Benign/BenignTraffic.csv"
df_tstat = pd.read_csv(ruta_tstat)




CSV encontrados: 4
BenignTraffic.pcap_Flow.csv
BenignTraffic1.pcap_Flow.csv
BenignTraffic2.pcap_Flow.csv
BenignTraffic3.pcap_Flow.csv
BenignTraffic.pcap_Flow.csv -> (183630, 85)
BenignTraffic1.pcap_Flow.csv -> (84526, 85)
BenignTraffic2.pcap_Flow.csv -> (91279, 85)
BenignTraffic3.pcap_Flow.csv -> (38895, 85)

Shape total: (398330, 85)
Número de columnas: 85
['Flow ID', 'Src IP', 'Src Port', 'Dst IP', 'Dst Port', 'Protocol', 'Timestamp', 'Flow Duration', 'Total Fwd Packet', 'Total Bwd packets', 'Total Length of Fwd Packet', 'Total Length of Bwd Packet', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max',

/tmp/ipykernel_18058/2552608385.py:31: DtypeWarning: Columns (113,116,117,127) have mixed types. Specify dtype option on import or set low_memory=False.
  df_tstat = pd.read_csv(ruta_tstat)


In [11]:
print(df_tstat.columns.tolist())

['c_iscrypto', 's_iscrypto', 's_pkts_all', 'c_isint', 's_isint', 'c_port', 'fqdn', 's_ip', 'c_pkts_all', 'c_ip', 'c_bytes_all', 's_port', 's_bytes_all', 'c_rst_cnt', 'c_ack_cnt', 'c_ack_cnt_p', 'c_bytes_uniq', 'c_pkts_data', 'c_pkts_retx', 'c_bytes_retx', 'c_pkts_ooo', 'c_syn_cnt', 'c_fin_cnt', 's_rst_cnt', 's_ack_cnt', 's_ack_cnt_p', 's_bytes_uniq', 's_pkts_data', 's_pkts_retx', 's_bytes_retx', 's_pkts_ooo', 's_syn_cnt', 's_fin_cnt', 'first', 'last', 'durat', 'c_first', 's_first', 'c_last', 's_last', 'c_first_ack', 's_first_ack', 'con_t', 'p2p_t', 'http_t', 'c_rtt_avg', 'c_rtt_min', 'c_rtt_max', 'c_rtt_std', 'c_rtt_cnt', 'c_ttl_min', 'c_ttl_max', 's_rtt_avg', 's_rtt_min', 's_rtt_max', 's_rtt_std', 's_rtt_cnt', 's_ttl_min', 's_ttl_max', 'p2p_st', 'ed2k_data', 'ed2k_sig', 'ed2k_c2s', 'ed2k_c2c', 'ed2k_chat', 'c_f1323_opt', 'c_tm_opt', 'c_win_scl', 'c_sack_opt', 'c_sack_cnt', 'c_mss', 'c_mss_max', 'c_mss_min', 'c_win_max', 'c_win_min', 'c_win_0', 'c_cwin_max', 'c_cwin_min', 'c_cwin_ini',

In [13]:
df_tstat["first"] = pd.to_datetime(df_tstat["first"], unit="s", errors="coerce")
df_tstat["last"] = pd.to_datetime(df_tstat["last"], unit="s", errors="coerce")

df_tstat["durat_sec"] = df_tstat["durat"]

In [14]:
df_tstat = df_tstat.rename(columns={
    "c_ip": "Src IP",
    "s_ip": "Dst IP",
    "c_port": "Src Port",
    "s_port": "Dst Port",
    "first": "Timestamp"
})

In [15]:
df_cic["Timestamp"] = pd.to_datetime(df_cic["Timestamp"])
df_cic["Duration_sec"] = df_cic["Flow Duration"] / 1e6

/tmp/ipykernel_18058/3204134152.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_cic["Timestamp"] = pd.to_datetime(df_cic["Timestamp"])


In [16]:
cols_id = ["Src IP", "Dst IP", "Src Port", "Dst Port"]

df_cic["flow_id"] = df_cic[cols_id].astype(str).agg("_".join, axis=1)
df_tstat["flow_id"] = df_tstat[cols_id].astype(str).agg("_".join, axis=1)

In [17]:
df_merge = pd.merge(
    df_cic,
    df_tstat,
    on="flow_id",
    how="inner",
    suffixes=("_cic", "_tstat")
)

In [18]:
# diferencia de tiempo
df_merge["time_diff"] = (
    df_merge["Timestamp_cic"] - df_merge["Timestamp_tstat"]
).abs().dt.total_seconds()

# diferencia de duración
df_merge["duration_diff"] = abs(
    df_merge["Duration_sec"] - df_merge["durat_sec"]
)

# filtro final
df_match = df_merge[
    (df_merge["time_diff"] < 1) &        # 1 segundo
    (df_merge["duration_diff"] < 0.5)    # medio segundo
]

In [19]:
print("Matches:", len(df_match))
print("Total CIC:", len(df_cic))
print("Porcentaje:", len(df_match)/len(df_cic)*100)

Matches: 0
Total CIC: 398330
Porcentaje: 0.0
